# 🎤 Voice-to-Voice RAG Assistant (Open Source)
Voice → STT (Whisper) → RAG → LLM → TTS → Voice
UI built with Gradio.

## Install Dependencies

In [ ]:
!pip install -q gradio faster-whisper TTS langchain langchain-community chromadb sentence-transformers transformers accelerate

## Import Libraries

In [ ]:
import gradio as gr
import tempfile
from faster_whisper import WhisperModel
from TTS.api import TTS
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

## Load Models

In [ ]:
whisper_model = WhisperModel('base')
tts = TTS(model_name='tts_models/en/ljspeech/tacotron2-DDC')

## Create Knowledge Base

In [ ]:
documents=[
'Retrieval Augmented Generation combines LLMs with external knowledge.',
'Vector databases store embeddings for semantic search.',
'Agentic RAG improves reasoning using tools and planning.',
'LangGraph enables multi-agent workflows.'
]

## Create Vector Database

In [ ]:
embedding=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore=Chroma.from_texts(documents,embedding)
retriever=vectorstore.as_retriever()

## Load Open Source LLM

In [ ]:
pipe=pipeline('text-generation',model='mistralai/Mistral-7B-Instruct-v0.1',max_new_tokens=200)
llm=HuggingFacePipeline(pipeline=pipe)

## Speech to Text

In [ ]:
def speech_to_text(audio):
 segments,_=whisper_model.transcribe(audio)
 text=''
 for seg in segments:
  text+=seg.text
 return text

## RAG Answer

In [ ]:
def rag_answer(q):
 docs=retriever.invoke(q)
 ctx='\n'.join([d.page_content for d in docs])
 prompt=f"Context:\n{ctx}\n\nQuestion:{q}\nAnswer:"
 return llm.invoke(prompt)

## Text to Speech

In [ ]:
def text_to_speech(text):
 file=tempfile.NamedTemporaryFile(delete=False,suffix='.wav').name
 tts.tts_to_file(text=text,file_path=file)
 return file

## Voice RAG Pipeline

In [ ]:
def voice_rag(audio):
 if audio is None:
  return 'Please record audio',None
 query=speech_to_text(audio)
 ans=rag_answer(query)
 voice=text_to_speech(ans)
 return ans,voice

## Gradio UI

In [ ]:
with gr.Blocks() as demo:
 gr.Markdown('# 🎤 Voice-to-Voice RAG Assistant')
 audio_in=gr.Audio(sources=['microphone'],type='filepath')
 text_out=gr.Textbox(label='AI Response')
 audio_out=gr.Audio(label='Voice Response')
 btn=gr.Button('Ask')
 btn.click(voice_rag,audio_in,[text_out,audio_out])
demo.launch()